# 01: Ingest Transaction Stream  -  "From Raw Events to Bronze"
**Exam Coverage**: DE Associate (Auto Loader) + DE Professional (Streaming, Watermarks)
**Duration**: 40 minutes
---
## Learning Objectives
By the end of this notebook, you will be able to:
- Configure Auto Loader for streaming JSON ingestion with rescued data capture
- Deduplicate payment gateway retries using watermarked dedup
- Build a robust Bronze transaction table
---

## Section 1: Ingesting Raw Payments
ModernPaymentsABC receives transaction data as JSON files via a payment gateway. These files are landed in a cloud volume. To ingest these files incrementally, we use **Auto Loader** (`cloudFiles`).

### Key Concepts
- **Auto Loader**: Efficiently ingests new files as they arrive in cloud storage.
- **Rescued Data Column**: Captures records that don't match the schema (e.g., malformed JSON, type mismatches) so they aren't lost.
- **Schema Inference**: Automatically detects the structure of your JSON data.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import * 

In [0]:
%run ./variables

---
### 🎯 EXERCISE 1: Configure Auto Loader with Rescue
**Your task**: Set up a streaming DataFrame to read JSON files from the transactions volume using Auto Loader.

**Requirements:**
- Define the transaction schema using `StructType`. Refer to **`data_contract.md`** in the project root for field names and types.
- Use the `cloudFiles` format with `json` as the source format.
- Provide the schema to the reader.
- Enable the rescued data column to capture malformed records (name it `_rescued_data`).
- Define the schema location for Auto Loader in the `_system` volume (use `CHECKPOINT_PATH + "/schema_bronze"`).

**Hint**: Make sure to use the `TRANSACTIONS_RAW_PATH` variable defined in `variables.py`.

In [0]:
trans_schema=StructType([
    StructField("txn_id",StringType(),True),
    StructField("customer_id",StringType(), True),
    StructField("merchant_id",StringType(),True),
    StructField("amount",DoubleType(),True),
    StructField("currency",StringType(),True),
    StructField("channel", StringType(), True),
    StructField("ip_country",StringType(),True),
    StructField("timestamp",TimestampType(),True)
])


raw_transaction_df=(spark.readStream.format("cloudFiles")
                    .option("cloudFiles.format","json")
                    .schema(trans_schema)
                    .option("cloudFiles.schemaLocation",f"{CHECKPOINT_PATH}/schema_loc")
                    .option("cloudFiles.rescuedDataColumn","_rescued_date")
                    .load(TRANSACTIONS_RAW_PATH))


                    

In [0]:
display(raw_transaction_df, checkpointLocation=CHECKPOINT_PATH + "/display_raw_txns")

In [0]:
print(TRANSACTIONS_RAW_PATH)
display(dbutils.fs.ls(TRANSACTIONS_RAW_PATH))

---
### 🎯 EXERCISE 2: Inspect Rescued Data
**Your task**: Query the `_rescued_data` column to see if any records were malformed during ingestion.

**Requirements:**
- Filter the `raw_txns_df` for records where `_rescued_data` is NOT null.
- Display the count and the content of the rescued data.

**Hint**: In a real production environment, you would monitor this count to detect upstream schema changes or integration bugs.

In [0]:
rescued_df=raw_transaction_df.filter("_rescued_date IS NOT NULL")

display(rescued_df,checkpointLocation=CHECKPOINT_PATH + "/display_rescued")

Section 2: Technical Deduplication

In Fintech, "at-least-once" delivery from payment gateways often results in duplicate records (retries). These are technical duplicates (same txn_id) and should be handled at the Bronze layer before business logic is applied.

Key Concepts
Watermarks: A threshold that tells Spark how long to wait for late-arriving data.
Deduplication within Watermark: Removes duplicates based on specific columns within a time window.

---
### 🎯 EXERCISE 3: Deduplicate Payment Retries
**Your task**: Use a watermark and `dropDuplicatesWithinWatermark` to ensure each `txn_id` is only processed once.

**Requirements:**
- Add a watermark of 5 minutes on the `timestamp` column.
- Remove duplicates based on the transaction ID (`txn_id`) within that watermark.
- This ensures that if the same transaction arrives twice within 5 minutes, only the first one is kept.

**Hint**: Use the streaming deduplication function that works with watermarks.

In [0]:
deduped_txns_df = (raw_transaction_df
 .withWatermark("timestamp", "5 minutes")
 .dropDuplicatesWithinWatermark(["txn_id"]))

---
### 🎯 EXERCISE 4: Write Bronze Transaction Table
**Your task**: Write the deduplicated stream to a Bronze Delta table.

**Requirements:**
- Add a column `_ingestion_timestamp` for the current ingestion time.
- Write to the Delta table defined in `TABLE_BRONZE_TRANSACTIONS`.
- Use "append" output mode.
- Define a checkpoint location in the `_system` volume (use `CHECKPOINT_PATH + "/bronze_txns"`).
- Use a trigger to process all available data in one batch (`availableNow=True`).

**Hint**: Trigger `availableNow=True` is the senior way to process all available data and then stop.

In [0]:
(deduped_txns_df
 .withColumn("_ingestion_timestamp", F.current_timestamp())
 .writeStream
 .format("delta")
 .outputMode("append")
 .option("checkpointLocation", CHECKPOINT_PATH + "/bronze_txns")
 .trigger(availableNow=True)
 .toTable(TABLE_BRONZE_TRANSACTIONS)
 .awaitTermination())


In [0]:
print(TABLE_BRONZE_TRANSACTIONS)

In [0]:
%sql

select * from Payment_monitoring.fintech.transactions_bronze where _rescued_date is not null

### 🔍 Verify Your Results
Run the cell below to inspect your output. You should see:
- **Row count**: ~200+ rows (depends on how many batches you generated  -  each `generate_batch` call adds ~100 transactions).
- **Key columns**: `txn_id`, `customer_id`, `merchant_id`, `amount`, `currency`, `timestamp`, `_rescued_data`, `_ingestion_timestamp`.
- **`_rescued_data`**: Most rows should show `null` here  -  meaning they parsed cleanly. Any non-null values indicate records with unexpected fields.
- **`_ingestion_timestamp`**: Every row should have a timestamp from when YOU ran the pipeline  -  this is your lineage metadata for auditing.

**What this means**: Your Bronze layer is now a faithful, append-only record of every payment event  -  including retries that were deduplicated and malformed records that were captured (not dropped). This is the foundation every downstream layer depends on.

In [0]:
 # 🔍 Verification query
bronze_result = spark.read.table(TABLE_BRONZE_TRANSACTIONS)
print(f"✅ {TABLE_BRONZE_TRANSACTIONS}: {bronze_result.count()} rows")
display(bronze_result.limit(10))

In [0]:
from pyspark.sql import functions as F

display(bronze_result.orderBy(F.col("txn_id").asc()
))

In [0]:
print("hello world")

In [0]:
display(spark.table("payment_monitoring.fintech.transactions_bronze"))